In [0]:
CATALOG = 'SALES'
SCHEMA =  'SILVER'
TARGET_TABLE = 'GEOLOCATIONS_ENRICHED'

In [0]:
geo_df = spark.table('SALES.BRONZE.GEOLOCATIONS') 
geo_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

geo_enriched_df = geo_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    upper(trim(col("geolocation_city"))).alias("geolocation_city"),
    upper(trim(col("geolocation_state"))).alias("geolocation_state"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
)

display(geo_enriched_df.limit(1))

In [0]:
geo_enriched_df.createOrReplaceTempView('enriched_geolocation_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY (geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state)
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_geolocation_temp_vw
  )
  WHERE rn = 1
) src
ON (src.geolocation_zip_code_prefix = tgt.geolocation_zip_code_prefix
    AND src.geolocation_lat = tgt.geolocation_lat
    AND src.geolocation_lng = tgt.geolocation_lng
    AND src.geolocation_city = tgt.geolocation_city
    AND src.geolocation_state = tgt.geolocation_state)
WHEN MATCHED THEN
  UPDATE SET
    tgt.geolocation_zip_code_prefix = src.geolocation_zip_code_prefix,
    tgt.geolocation_lat = src.geolocation_lat,
    tgt.geolocation_lng = src.geolocation_lng,
    tgt.geolocation_city = src.geolocation_city,
    tgt.geolocation_state = src.geolocation_state
WHEN NOT MATCHED THEN
  INSERT (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.geolocation_zip_code_prefix,
    src.geolocation_lat,
    src.geolocation_lng,
    src.geolocation_city,
    src.geolocation_state,
    src.file_path,
    current_timestamp()
  )
""")